# DINO Food Pretraining: SmartRecipe+ Unsupervised Component

**Workflow:**
1. Mount Google Drive (checkpoint persistence across disconnects)
2. Clone repo & install dependencies
3. Verify data pipeline
4. Train ResNet-50 (~4-6 hrs)
5. Train ViT-S/16 (~4-6 hrs)
6. Export weights & run evaluation

## 1. GPU Check & Drive Mount

In [2]:
# Verify GPU is available
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# Mount Google Drive 
from google.colab import drive
drive.mount('/content/drive')

import os

# DRIVE_BASE matches the auto-save path in pretrain.py:
#   save_to_drive → /content/drive/MyDrive/dino_checkpoints/{backbone}/epoch_XXXX.pth
DRIVE_BASE = '/content/drive/MyDrive/dino_checkpoints'
DRIVE_EXPORTS = f'{DRIVE_BASE}/exports'

os.makedirs(f'{DRIVE_BASE}/resnet50', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/vit_small_patch16_224', exist_ok=True)
os.makedirs(DRIVE_EXPORTS, exist_ok=True)
print(f'Checkpoint dir: {DRIVE_BASE}')
print(f'Exports dir:    {DRIVE_EXPORTS}')

## 2. Clone Repo & Install Dependencies

In [ ]:
# Clone repo
%cd /content
!rm -rf smartrecipeplus  # clean slate on reconnect
!git clone https://github.com/jeanvelezdata/smartrecipeplus.git
%cd /content/smartrecipeplus/unsupervised

In [ ]:
# Install dependencies
!pip install -q timm datasets PyYAML tqdm scikit-learn

## 3. Verify Data Pipeline

In [ ]:
# Download Food-101 and verify
!python data/prepare_food101.py --verify

In [ ]:
import os, torchvision

DATA_DIR = '/content/food101_images'

if os.path.isdir(DATA_DIR) and len(os.listdir(DATA_DIR)) > 100_000:
    print(f'Already have images at {DATA_DIR}, skipping download.')
else:
    train_ds = torchvision.datasets.Food101(root='/content/food101_raw', split='train',  download=True)
    val_ds   = torchvision.datasets.Food101(root='/content/food101_raw', split='test',   download=True)

    os.makedirs(DATA_DIR, exist_ok=True)
    idx = 0
    for ds in (train_ds, val_ds):
        for img, _ in ds:
            img.save(f'{DATA_DIR}/{idx:06d}.jpg')
            idx += 1
        print(f'  ... {idx:,} images saved so far')

    print(f'Done, {idx:,} images saved to {DATA_DIR}')

print(f'Image count: {len(os.listdir(DATA_DIR)):,}')

In [ ]:
import glob, os

# pretrain.py saves checkpoints
drive_ckpts = sorted(glob.glob(f'{DRIVE_BASE}/resnet50/epoch_*.pth'))
resume_flag = f'--resume {drive_ckpts[-1]}' if drive_ckpts else ''

if resume_flag:
    print(f'Resuming from: {drive_ckpts[-1]}')
else:
    print('Starting fresh, no previous checkpoint found.')

In [ ]:
# Train ResNet-50 with DINO 
!python pretrain.py \
    --config configs/resnet50.yaml \
    --data-dir {DATA_DIR} \
    {resume_flag}

In [ ]:
import shutil

local_ckpts = sorted(glob.glob('./checkpoints/resnet50/epoch_*.pth'))
if local_ckpts:
    dest = f'{DRIVE_BASE}/resnet50/{os.path.basename(local_ckpts[-1])}'
    shutil.copy2(local_ckpts[-1], dest)
    print(f'Backed up to Drive: {dest}')
else:
    print('No local checkpoints found.')

In [ ]:
# Copy final ResNet-50 checkpoint to Drive
import shutil
local_ckpts = sorted(glob.glob('./checkpoints/resnet50/checkpoint_*.pth'))
if local_ckpts:
    dest = f'{DRIVE_BASE}/resnet50/{os.path.basename(local_ckpts[-1])}'
    shutil.copy2(local_ckpts[-1], dest)
    print(f'Saved to Drive: {dest}')

In [ ]:
# cfg["backbone"] for ViT is "vit_small_patch16_224"
drive_ckpts_vit = sorted(glob.glob(f'{DRIVE_BASE}/vit_small_patch16_224/epoch_*.pth'))
resume_flag_vit = f'--resume {drive_ckpts_vit[-1]}' if drive_ckpts_vit else ''

if resume_flag_vit:
    print(f'Resuming from: {drive_ckpts_vit[-1]}')
else:
    print('Starting fresh — no previous checkpoint found.')

In [ ]:
# Train ViT-Small/16 with DINO (local images — fast path)
!python pretrain.py \
    --config configs/vit_small16.yaml \
    --data-dir {DATA_DIR} \
    {resume_flag_vit}

In [ ]:
local_ckpts_vit = sorted(glob.glob('./checkpoints/vit_small_patch16_224/epoch_*.pth'))
if local_ckpts_vit:
    dest = f'{DRIVE_BASE}/vit_small_patch16_224/{os.path.basename(local_ckpts_vit[-1])}'
    shutil.copy2(local_ckpts_vit[-1], dest)
    print(f'Backed up to Drive: {dest}')
else:
    print('No local checkpoints found.')

In [ ]:
# Copy final ViT checkpoint to Drive
local_ckpts_vit = sorted(glob.glob('./checkpoints/vit_small16/checkpoint_*.pth'))
if local_ckpts_vit:
    dest = f'{DRIVE_BASE}/vit_small16/{os.path.basename(local_ckpts_vit[-1])}'
    shutil.copy2(local_ckpts_vit[-1], dest)
    print(f'Saved to Drive: {dest}')

In [ ]:
resnet_ckpts = sorted(glob.glob(f'{DRIVE_BASE}/resnet50/epoch_*.pth'))
vit_ckpts    = sorted(glob.glob(f'{DRIVE_BASE}/vit_small_patch16_224/epoch_*.pth'))

if not resnet_ckpts:
    raise FileNotFoundError(f'No ResNet-50 checkpoints found in {DRIVE_BASE}/resnet50/')
if not vit_ckpts:
    raise FileNotFoundError(f'No ViT checkpoints found in {DRIVE_BASE}/vit_small_patch16_224/')

resnet_ckpt = resnet_ckpts[-1]
vit_ckpt    = vit_ckpts[-1]
print(f'ResNet-50 checkpoint: {resnet_ckpt}')
print(f'ViT-S/16 checkpoint:  {vit_ckpt}')

In [ ]:
# Export ResNet-50
!python export_weights.py \
    --checkpoint {resnet_ckpt} \
    --config configs/resnet50.yaml \
    --output {DRIVE_EXPORTS}/resnet50_food_dino.pth

# Export ViT-S/16
!python export_weights.py \
    --checkpoint {vit_ckpt} \
    --config configs/vit_small16.yaml \
    --output {DRIVE_EXPORTS}/vit_small16_food_dino.pth

print(f'\nExported weights saved to: {DRIVE_EXPORTS}')
!ls -lh {DRIVE_EXPORTS}/

In [ ]:
# Export ResNet-50
!python export_weights.py \
    --checkpoint {resnet_ckpt} \
    --config configs/resnet50.yaml \
    --output {DRIVE_BASE}/exports/resnet50_food_dino.pth

# Export ViT-S/16
!python export_weights.py \
    --checkpoint {vit_ckpt} \
    --config configs/vit_small16.yaml \
    --output {DRIVE_BASE}/exports/vit_small16_food_dino.pth

print(f'\nExported weights saved to: {DRIVE_BASE}/exports/')
!ls -lh {DRIVE_BASE}/exports/

## 7. Evaluate through Linear Probe

In [ ]:
# Linear probe: ResNet-50
!python evaluate.py \
    --checkpoint {resnet_ckpt} \
    --config configs/resnet50.yaml \
    --eval-type linear_probe

In [ ]:
# Linear probe: ViT-S/16
!python evaluate.py \
    --checkpoint {vit_ckpt} \
    --config configs/vit_small16.yaml \
    --eval-type linear_probe

## 8. Evaluate through Visualizations

In [ ]:
# t-SNE visualization
!python evaluate.py \
    --checkpoint {resnet_ckpt} \
    --config configs/resnet50.yaml \
    --eval-type tsne

# Display the plot inline
from IPython.display import Image, display
display(Image('tsne_visualization.png'))

In [ ]:
for f in ['tsne_visualization.png', 'attention_maps.png']:
    if os.path.exists(f):
        shutil.copy2(f, f'{DRIVE_EXPORTS}/{f}')
        print(f'Saved {f} to Drive')

# Copy training logs from both runs
for log in ['./checkpoints/resnet50/training_log.csv', './checkpoints/vit_small_patch16_224/training_log.csv']:
    if os.path.exists(log):
        name = os.path.basename(os.path.dirname(log)) + '_training_log.csv'
        shutil.copy2(log, f'{DRIVE_EXPORTS}/{name}')
        print(f'Saved {name} to Drive')

print(f'\nAll outputs in: {DRIVE_EXPORTS}')
!ls -lh {DRIVE_EXPORTS}/

In [ ]:
# Copy evaluation outputs to Drive
for f in ['tsne_visualization.png', 'attention_maps.png', 'training_log.csv']:
    if os.path.exists(f):
        shutil.copy2(f, f'{DRIVE_BASE}/exports/{f}')
        print(f'Saved {f} to Drive')

print(f'\nAll outputs in: {DRIVE_BASE}/exports/')
!ls -lh {DRIVE_BASE}/exports/

In [ ]:
import torch
import torchvision.models as models
import timm

# Test ResNet-50 loading
resnet = models.resnet50()
state = torch.load(f'{DRIVE_EXPORTS}/resnet50_food_dino.pth', map_location='cpu')
resnet.load_state_dict(state, strict=False)
print(f'ResNet-50 loaded — {sum(p.numel() for p in resnet.parameters())/1e6:.1f}M params')

# Test ViT-S/16 loading
vit = timm.create_model('vit_small_patch16_224', pretrained=False)
state = torch.load(f'{DRIVE_EXPORTS}/vit_small16_food_dino.pth', map_location='cpu')
vit.load_state_dict(state, strict=False)
print(f'ViT-S/16 loaded — {sum(p.numel() for p in vit.parameters())/1e6:.1f}M params')

# Quick forward pass
dummy = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    print(f'ResNet-50 output shape: {resnet(dummy).shape}')
    print(f'ViT-S/16 output shape:  {vit(dummy).shape}')

print('\nPass.')

In [ ]:
import torch
import torchvision.models as models
import timm

# Test ResNet-50 loading
resnet = models.resnet50()
state = torch.load(f'{DRIVE_BASE}/exports/resnet50_food_dino.pth', map_location='cpu')
resnet.load_state_dict(state, strict=False)
print(f'ResNet-50 loaded — {sum(p.numel() for p in resnet.parameters())/1e6:.1f}M params')

# Test ViT-S/16 loading
vit = timm.create_model('vit_small_patch16_224', pretrained=False)
state = torch.load(f'{DRIVE_BASE}/exports/vit_small16_food_dino.pth', map_location='cpu')
vit.load_state_dict(state, strict=False)
print(f'ViT-S/16 loaded — {sum(p.numel() for p in vit.parameters())/1e6:.1f}M params')

# Quick forward pass
dummy = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    print(f'ResNet-50 output shape: {resnet(dummy).shape}')
    print(f'ViT-S/16 output shape:  {vit(dummy).shape}')

print('\nPass.')